# 03 — Stage 2 vs Stage 3
같은 (face, species, ip_scale, seed)에서 Stage 2 / Stage 3 결과를 위·아래로 붙여 본다.
Stage 3는 controlnet_scale을 한 값(기본 0.5)으로 고정해 비교.

In [ ]:
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
def latest(stage):
    runs = sorted(Path('experiments/runs').glob(f'stage{stage}_*'))
    return runs[-1]

RUN2 = latest(2); RUN3 = latest(3)
df2 = pd.read_parquet(RUN2 / 'metadata.parquet'); df2 = df2[df2['image_path']!='']
df3 = pd.read_parquet(RUN3 / 'metadata.parquet'); df3 = df3[df3['image_path']!='']
print('stage2:', RUN2.name, len(df2), '| stage3:', RUN3.name, len(df3))

In [ ]:
CN_FIX = 0.5
SEED = 0

shared_ip = sorted(set(df2['ip_scale'].unique()) & set(df3['ip_scale'].unique()))
print('shared ip_scale:', shared_ip)
faces   = sorted(set(df2['face_id'].unique()) & set(df3['face_id'].unique()))
species = sorted(set(df2['species'].unique()) & set(df3['species'].unique()))

for fid in faces:
    for sp in species:
        fig, axes = plt.subplots(2, len(shared_ip), figsize=(2.4*len(shared_ip), 4.8))
        if len(shared_ip) == 1: axes = [[axes[0]], [axes[1]]]
        for c, ip in enumerate(shared_ip):
            r2 = df2[(df2['face_id']==fid) & (df2['species']==sp) & (df2['ip_scale']==ip) & (df2['seed']==SEED)]
            r3 = df3[(df3['face_id']==fid) & (df3['species']==sp) & (df3['ip_scale']==ip) & (df3['controlnet_scale']==CN_FIX) & (df3['seed']==SEED)]
            for ax in (axes[0][c], axes[1][c]): ax.axis('off')
            if not r2.empty: axes[0][c].imshow(Image.open(r2.iloc[0]['image_path']))
            if not r3.empty: axes[1][c].imshow(Image.open(r3.iloc[0]['image_path']))
            if c == 0:
                axes[0][c].set_title(f'Stage 2  ip={ip:.1f}', loc='left')
                axes[1][c].set_title(f'Stage 3 cn={CN_FIX}', loc='left')
            else:
                axes[0][c].set_title(f'ip={ip:.1f}')
        fig.suptitle(f'{fid} / {sp}'); plt.tight_layout(); plt.show()